In [16]:
import pandas as pd
import numpy as np
from nautilus_trader.model import BarType
from nautilus_trader.persistence import ParquetDataCatalog
from nautilus_bridge.alphas.session_resampling import bar_row, bars_frame, resample_by_session
from nautilus_bridge.alphas.loader import load_alpha
from nautilus_bridge.data.catalog import catalog_path
from nautilus_bridge.data.trading_days import LOOKBACK_TRADING_DAYS, warmup_start
from nautilus_bridge.analyzer.forecast_statistics import forecast_statistics
from nautilus_bridge.alphas.session_resampling import bars_per_day
from nautilus_bridge.backtest.sample_split import sample_window
from methodica.dsl import compute

pd.options.plotting.backend = 'plotly'

### Active Porfolio Management (Vectorized Prototype)

In [17]:
from nautilus_trader.analysis import (
    SharpeRatio, SortinoRatio, ReturnsAverage, ReturnsVolatility, ProfitFactor,
    ReturnsAverageWin, ReturnsAverageLoss, TailRatio, ReturnsSkewness, ReturnsKurtosis,
    RiskReturnRatio, MaxDrawdown,
)
from nautilus_bridge.analyzer.analyzer import ordered_stats
from nautilus_bridge.analyzer.forecast_statistics import forecast_statistics
from nautilus_bridge.analyzer.statistics import NANOS_PER_YEAR, TRADING_DAYS_PER_YEAR
from nautilus_bridge.alphas.session_resampling import bars_per_day

RETURN_STATISTICS = [
    SharpeRatio, SortinoRatio, ReturnsAverage, ReturnsVolatility, ProfitFactor,
    ReturnsAverageWin, ReturnsAverageLoss, TailRatio, ReturnsSkewness, ReturnsKurtosis,
    RiskReturnRatio, MaxDrawdown,
]


def turnover_daily(contracts):
    traded = contracts.diff().fillna(contracts).abs().sum()
    ts = contracts.index.asi8
    dt = np.diff(ts, append=ts[-1])
    avg_abs_contracts = (contracts.abs() * dt).sum() / dt.sum()
    years = dt.sum() / NANOS_PER_YEAR
    return traded / 2 / avg_abs_contracts / years / TRADING_DAYS_PER_YEAR


def compute_statistics(bars, forecast_col, book_size, timeframe, multiplier, cost_per_contract):
    equity = (book_size + bars.net_pnl.fillna(0).cumsum()).tz_convert("UTC")
    days = pd.date_range(equity.index[0].normalize(), equity.index[-1].normalize(), freq="D", tz="UTC")
    daily_returns = (
        equity.groupby(equity.index.normalize()).last()
        .reindex(days).ffill().fillna(book_size)
        .pct_change().fillna(0)
    )
    returns = {int(ts.value): r for ts, r in daily_returns.items()}

    stats_returns = {stat().name: stat().calculate_from_returns(returns) for stat in RETURN_STATISTICS}
    stats_forecast = forecast_statistics(bars[forecast_col], bars.close, bars_per_day(timeframe), multiplier, cost_per_contract)
    ordered = list(ordered_stats(stats_returns, {}, stats_forecast).items())
    turnover_at = [name for name, _ in ordered].index("Returns Volatility (252 days)") + 1
    ordered.insert(turnover_at, ("Turnover (round trips per day)", turnover_daily(bars.contracts)))
    stats = {"Total Net PnL": bars.net_pnl.sum(), **dict(ordered)}

    return pd.Series({
        name: f"{float(value):,.0f}" if name == "Total Net PnL" else f"{float(value):,.4f}"
        for name, value in stats.items()
    })

In [18]:
instrument_id   = "VN30F1M.HNX"
timeframe       = "1-MINUTE"
start           = pd.Timestamp("2018-09-25", tz="UTC")
end             = pd.Timestamp("2026", tz="UTC")
book_size       = 100_000_000
commission      = 22_750
slippage        = 0.0  # index points per contract traded
multiplier      = 100_000
margin_init     = 0.2


src = bars_frame(
    bar_row(b)
    for b in ParquetDataCatalog(str(catalog_path())).query_bars([f"{instrument_id}-1-MINUTE-LAST-EXTERNAL"])
)
bars = resample_by_session(src, timeframe)

bars.index = pd.to_datetime(bars.index, unit="ns", utc=True).tz_convert("Asia/Ho_Chi_Minh")


In [19]:
tz            = "Asia/Ho_Chi_Minh"
horizon       = 1
vol_window    = 20 * bars_per_day(timeframe)
risk_aversion = 2.36
bps           = 10_000


def ema_cross(bars):
    close = bars.close
    fast = close.ewm(span=10, adjust=False, min_periods=10).mean()
    slow = close.ewm(span=20, adjust=False, min_periods=20).mean()
    price_std = close.diff().ewm(span=20, adjust=False, min_periods=20).std()
    return (fast - slow) / price_std

In [20]:
def fit_window():
    fit_start = pd.Timestamp("2020-01-01", tz="UTC")
    fit_end, _ = sample_window("IS")
    return fit_start, fit_end

def raw_signal(alpha_fn, tf, src, index):
    signal = alpha_fn(resample_by_session(src, tf))
    signal.index = pd.to_datetime(signal.index, unit="ns", utc=True).tz_convert(tz)
    return signal.reindex(index, method="ffill")

def compute_raw_signals(alphas, src, index):
    return pd.DataFrame({
        f"{alpha_fn.__name__}_{tf.lower().replace('-', '_')}": raw_signal(alpha_fn, tf, src, index)
        for alpha_fn, tf in alphas
    })

def standardize(signals, fit_start, fit_end):
    in_fit = (signals.index >= fit_start) & (signals.index < fit_end)
    mean, std = signals[in_fit].mean(), signals[in_fit].std()
    return (signals - mean) / std

def compute_combination_weights(standardized_signals, forward_return, train):
    ic  = standardized_signals[train].corrwith(forward_return[train])
    rho = standardized_signals[train].corr()
    weights = pd.Series(np.linalg.solve(rho, ic), index=standardized_signals.columns)
    return weights, ic, rho

def forecast_volatility(bars):
    return bars.close.pct_change(horizon).rolling(vol_window).std()

def refine_forecast(signals, weights, predicted_volatility):
    return predicted_volatility * (signals @ weights) * bps

def optimal_position(alpha_bps, predicted_volatility, risk_aversion):
    return (alpha_bps / bps) / (2 * risk_aversion * predicted_volatility ** 2)

def estimate_risk_aversion(bars):
    return risk_aversion

def compute_target_contracts(position, close, book_size, multiplier, margin_init):
    max_contracts = np.floor(book_size / (close * multiplier * margin_init))
    return (
        np.trunc(position * book_size / (close * multiplier))
        .clip(-max_contracts, max_contracts)
        .fillna(0)
        + 0.0
    )

def compute_pnl(contracts, close, multiplier, cost_per_contract):
    traded_contracts = contracts.diff().fillna(contracts)
    pnl  = contracts.shift(1) * close.diff() * multiplier + 0.0
    cost = traded_contracts.abs() * cost_per_contract
    return pd.DataFrame({
        "contracts": contracts,
        "traded_contracts": traded_contracts,
        "pnl": pnl,
        "cost": cost,
        "net_pnl": pnl - cost,
    })

def trade_only_in_window(contracts, test):
    contracts = contracts.where(test, 0.0)
    contracts.iloc[np.flatnonzero(test)[-1]] = 0.0
    return contracts


def backtest(
        alphas, 
        volatility_model, 
        risk_aversion_model, 
        src, 
        bars, 
        sample, 
        book_size, 
        multiplier, 
        margin_init, 
        commission, 
        slippage
):    
    start, end           = sample_window(sample)
    fit_start, fit_end   = fit_window()
    raw_signals          = compute_raw_signals(alphas, src, bars.index)
    standardized_signals = standardize(raw_signals, fit_start, fit_end)
    forward_return       = bars.close.pct_change(horizon).shift(-horizon)
    realized_at          = bars.index.to_series().shift(-horizon)
    train                = ((bars.index >= fit_start) & (realized_at < fit_end)).to_numpy()
    weights, ic, rho     = compute_combination_weights(standardized_signals, forward_return, train)
    in_fit               = (bars.index >= fit_start) & (bars.index < fit_end)
    fit = {
        "data": (
            bars.join(raw_signals)
            .join(standardized_signals.add_prefix("standardized_"))
            .assign(forward_return=forward_return, train=train)[in_fit]
        ),
        "ic": ic,
        "rho": rho,
        "weights": weights,
    }

    lookback_start       = warmup_start(BarType.from_str(f"{instrument_id}-1-MINUTE-LAST-EXTERNAL"), start, LOOKBACK_TRADING_DAYS)
    bars                 = bars[(bars.index >= lookback_start) & (bars.index < end)]
    raw_signals          = raw_signals.loc[bars.index]
    standardized_signals = standardized_signals.loc[bars.index]
    test                 = bars.index >= start

    vol                  = volatility_model(bars)
    alpha                = refine_forecast(standardized_signals, weights, vol)
    risk_aversion        = risk_aversion_model(bars)
    position             = optimal_position(alpha, vol, risk_aversion)
    contracts            = compute_target_contracts(position, bars.close, book_size, multiplier, margin_init)
    contracts            = trade_only_in_window(contracts, test)
    cost_per_contract    = commission + slippage * multiplier
    pnl                  = compute_pnl(contracts, bars.close, multiplier, cost_per_contract)
    results = bars.join(raw_signals).assign(vol=vol, alpha=alpha, position=position).join(pnl)[test]
    performance = compute_statistics(results, "alpha", book_size, timeframe, multiplier, cost_per_contract)
    return results, performance, fit

In [21]:
results, performance, fit = backtest(
    alphas=[(ema_cross, "30-MINUTE"), 
            # (ema_cross, "30-MINUTE")
    ],
    volatility_model=forecast_volatility,
    risk_aversion_model=estimate_risk_aversion,
    src=src,
    bars=bars,
    sample="IS",
    book_size=book_size,
    multiplier=multiplier,
    margin_init=margin_init,
    commission=commission,
    slippage=slippage,
)

In [22]:
performance

Total Net PnL                        64,667,500
Sharpe Ratio (Realized, 252 days)        1.6954
Sharpe Ratio (IC-based, 252 days)        0.2023
IC (Pearson, 1 bar)                      0.0013
Sortino Ratio (252 days)                 4.9777
Average (Return)                         0.0010
Returns Volatility (252 days)            0.1425
Turnover (round trips per day)           1.0457
Profit Factor                            2.1443
Average Win (Return)                     0.0150
Average Loss (Return)                   -0.0068
Tail Ratio                               1.4355
Returns Skewness                         5.5246
Returns Kurtosis                        43.9018
Risk Return Ratio                        0.1068
Max Drawdown                            -0.0742
dtype: str

In [23]:
results.position.describe()

count    88561.000000
mean         0.086413
std          1.457436
min         -6.778400
25%         -0.724688
50%          0.174234
75%          0.829893
max          4.869162
Name: position, dtype: float64

## Monte Carlo Simulation

In [24]:
import numpy as np

def simulate_max_drawdown(annual_return, annual_vol, days=252, n_paths=20_000, seed=0):
    rng = np.random.default_rng(seed)
    daily = rng.normal(annual_return / days, annual_vol / np.sqrt(days), size=(n_paths, days))
    equity = np.concatenate([np.ones((n_paths, 1)), 1 + np.cumsum(daily, axis=1)], axis=1)
    return (np.maximum.accumulate(equity, axis=1) - equity).max(axis=1)

def information_ratio(expected_return, expected_volatility):
    return expected_return / expected_volatility

def drawdown_profile(expected_return, max_drawdown, vols):
    rows = []
    for vol in vols:
        mdd = simulate_max_drawdown(expected_return, vol)
        rows.append({
            "volatility": vol,
            "information_ratio": expected_return / vol,
            "median_max_drawdown": np.median(mdd),
            "prob_within_max_drawdown": np.mean(mdd < max_drawdown),
        })
    return pd.DataFrame(rows).set_index("volatility")


def max_volatility_for_drawdown(profile, probability):
    feasible = profile.index[profile.prob_within_max_drawdown >= probability]
    return feasible.max() if len(feasible) else None

def required_information_coefficient(information_ratio, bars_per_year, horizon):
    breadth = bars_per_year / horizon
    return information_ratio / np.sqrt(breadth)

def required_alpha_ic(combined_ic, n_alphas, max_correlation):
    if np.isinf(n_alphas):
        multiplier = 1 / np.sqrt(max_correlation)
    else:
        multiplier = np.sqrt(n_alphas / (1 + (n_alphas - 1) * max_correlation))
    return combined_ic / multiplier

def risk_aversion_from_target(expected_information_ratio, expected_volatility):
    return expected_information_ratio / (2 * expected_volatility)

In [25]:
# expected_return = 1.0
# max_drawdown    = 0.2
# max_correlation = 0.5

# profile                    = drawdown_profile(expected_return=expected_return, max_drawdown=max_drawdown, vols=np.arange(0.05, 1.51, 0.01))
# expected_volatility        = max_volatility_for_drawdown(profile, probability=0.85)
# expected_information_ratio = information_ratio(expected_return, expected_volatility)
# risk_aversion              = risk_aversion_from_target(expected_information_ratio, expected_volatility)

# required_ic_combined = required_information_coefficient(
#     information_ratio=expected_information_ratio,
#     bars_per_year=bars_per_day(timeframe) * TRADING_DAYS_PER_YEAR,
#     horizon=5,
# )
# required_ic_single = required_alpha_ic(required_ic_combined, n_alphas=15, max_correlation=max_correlation)

# print("Required IC combined: ", required_ic_combined)
# print("Required IC single: ", required_ic_single)
# print("Risk aversion: ", risk_aversion)

In [26]:
# expected_volatility

### Active Portfolio Management (DSL Prototype)

In [27]:
def raw_signal(expression, tf, src, index):
    signal = compute(expression, resample_by_session(src, tf))
    signal.index = pd.to_datetime(signal.index, unit="ns", utc=True).tz_convert(tz)
    return signal.reindex(index, method="ffill")

def compute_raw_signals(alphas, src, index):
    return pd.DataFrame({
        name: raw_signal(expression, tf, src, index)
        for name, (expression, tf) in alphas.items()
    })

In [38]:
ema_cross_5m      = ("(ewma(close, 10) - ewma(close, 20)) / ewm_std(ts_delta(close, 1), 20)", "5-MINUTE")
momentum_5m        = ("ts_delta(close, 20) / ewm_std(ts_delta(close, 1), 20)", "5-MINUTE")
reversal_1m        = ("-ts_delta(close, 3) / ewm_std(ts_delta(close, 1), 20)", "1-MINUTE")
zscore_revert_5m   = ("-(close - ts_mean(close, 60)) / ts_std(close, 60)", "5-MINUTE")
channel_pos_15m    = ("(close - ts_min(low, 40)) / (ts_max(high, 40) - ts_min(low, 40)) - 0.5", "15-MINUTE")
macd_hist_15m      = ("(ewma(close, 12) - ewma(close, 26) - ewma(ewma(close, 12) - ewma(close, 26), 9)) / ewm_std(ts_delta(close, 1), 26)", "15-MINUTE")
rsi_like_5m        = ("ewma(ts_delta(close, 1), 14) / ewma(abs(ts_delta(close, 1)), 14)", "5-MINUTE")
bar_revert_5m      = ("-ewma((2 * close - high - low) / (high - low + 0.1), 5)", "5-MINUTE")
volume_momentum_5m = ("ts_delta(close, 10) / ewm_std(ts_delta(close, 1), 20) * volume / ts_mean(volume, 60)", "5-MINUTE")
price_volume_30m   = ("ts_corr(ts_delta(close, 1), volume, 20)", "30-MINUTE")

alphas = {
    "ema_cross_5m":      ema_cross_5m,
    "momentum_5m":        momentum_5m,
    "reversal_1m":        reversal_1m,
    "zscore_revert_5m":   zscore_revert_5m,
    "channel_pos_15m":    channel_pos_15m,
    "macd_hist_15m":      macd_hist_15m,
    "rsi_like_5m":        rsi_like_5m,
    "bar_revert_5m":      bar_revert_5m,
    "volume_momentum_5m": volume_momentum_5m,
    "price_volume_30m":   price_volume_30m,
}


results, performance, fit = backtest(
    alphas=alphas,
    volatility_model=forecast_volatility,
    risk_aversion_model=estimate_risk_aversion,
    src=src,
    bars=bars,
    sample="IS",
    book_size=book_size,
    multiplier=multiplier,
    margin_init=margin_init,
    commission=commission,
    slippage=slippage,
)
performance

Total Net PnL                        -465,925,000
Sharpe Ratio (Realized, 252 days)          0.6622
Sharpe Ratio (IC-based, 252 days)         -2.0004
IC (Pearson, 1 bar)                        0.0157
Sortino Ratio (252 days)                   1.2962
Average (Return)                           0.0070
Returns Volatility (252 days)              2.6753
Turnover (round trips per day)            28.9044
Profit Factor                              1.5131
Average Win (Return)                       0.0480
Average Loss (Return)                     -0.0565
Tail Ratio                                 0.8861
Returns Skewness                           9.4417
Returns Kurtosis                         217.9093
Risk Return Ratio                          0.0417
Max Drawdown                              -4.4927
dtype: str

In [39]:
fit['data']

,open,high,low,close,volume,ema_cross_5m,momentum_5m,reversal_1m,zscore_revert_5m,channel_pos_15m,...,standardized_reversal_1m,standardized_zscore_revert_5m,standardized_channel_pos_15m,standardized_macd_hist_15m,standardized_rsi_like_5m,standardized_bar_revert_5m,standardized_volume_momentum_5m,standardized_price_volume_30m,forward_return,train
bar_end,,,,,,,,,,,,,,,,,,,,,
2020-01-02 09:01:00+07:00,877.5,877.6,877.3,877.3,433.0,-0.075544,-1.601382,-0.000000,1.319186,0.004505,...,0.008806,1.004577,-0.189000,-0.351839,-0.581800,-0.298418,-0.764123,0.536970,-0.000228,True
2020-01-02 09:02:00+07:00,877.3,877.3,877.1,877.1,135.0,-0.075544,-1.601382,0.681457,1.319186,0.004505,...,0.431022,1.004577,-0.189000,-0.351839,-0.581800,-0.298418,-0.764123,0.536970,0.000114,True
2020-01-02 09:03:00+07:00,877.1,877.2,877.1,877.2,181.0,-0.075544,-1.601382,-1.609329,1.319186,0.004505,...,-0.988299,1.004577,-0.189000,-0.351839,-0.581800,-0.298418,-0.764123,0.536970,0.000228,True
2020-01-02 09:04:00+07:00,877.2,877.4,877.2,877.4,128.0,-0.075544,-1.601382,-0.186844,1.319186,0.004505,...,-0.106959,1.004577,-0.189000,-0.351839,-0.581800,-0.298418,-0.764123,0.536970,-0.000114,True
2020-01-02 09:05:00+07:00,877.4,877.6,877.3,877.3,557.0,-0.050196,0.119089,-0.391833,0.401358,0.004505,...,-0.233965,0.355983,-0.189000,-0.351839,-0.006147,0.033220,0.070739,0.536970,-0.000114,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-11-15 14:27:00+07:00,1280.6,1280.9,1279.7,1280.5,1224.0,-0.321619,0.537921,-0.955333,0.940992,-0.381690,...,-0.583098,0.737322,-1.426806,0.267773,-0.271701,-0.482520,-0.800376,1.420573,0.000547,True
2024-11-15 14:28:00+07:00,1280.5,1281.5,1280.5,1281.2,1403.0,-0.321619,0.537921,-0.000000,0.940992,-0.381690,...,0.008806,0.737322,-1.426806,0.267773,-0.271701,-0.482520,-0.800376,1.420573,0.000468,True
2024-11-15 14:29:00+07:00,1281.2,1282.3,1281.2,1281.8,1916.0,-0.321619,0.537921,-1.118370,0.940992,-0.381690,...,-0.684112,0.737322,-1.426806,0.267773,-0.271701,-0.482520,-0.800376,1.420573,-0.001404,True
